<a href="https://colab.research.google.com/github/Anas-Mirza/Practice-GGUF-Colab/blob/main/awesome_koboldcpp_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Awesome KoboldCpp / llama.cpp Colab Notebook

Adapted and extended from OneClick_LLM_API_onColab by Seyf EL
 – thank you!
Original source: https://github.com/seyf1elislam/OneClick_LLM_API_onColab

Used for experimenting with GGUF models like Nemotron-Nano-12B on free Colab tier.


# $\color{#19ABEA}{\text{Run any gguf quantized models in Latest Version KoboldCpp on Colab :}}$
## This is modified version  KoboldCpp Colab Notebook

It's really easy to get started. Just press the two **Play** buttons below, and then connect to the **Cloudflare URL** shown at the end.
You can select a model from the dropdown, or enter a **custom repo name** l (Example: `unsloth/Qwen3-8B-GGUF` with quant `Q5_k_m`)

**Keep this page open and occationally check for captcha's so that your AI is not shut down**

> Notebook rep : [Notebook github Repository](https://github.com/seyf1elislam/LocalLLM_OneClick_Colab).   
> koboldcpp : [https://github.com/LostRuins/koboldcpp](https://github.com/LostRuins/koboldcpp)

In [ ]:
#@title <-- Tap this if you play on Mobile { display-mode: "form" }
%%html
<b>Press play on the music player to keep the tab alive, then start KoboldCpp below</b><br/>
<audio autoplay="" src="https://raw.githubusercontent.com/KoboldAI/KoboldAI-Client/main/colab/silence.m4a" loop controls>


#  $\color{#19ABEA}{Run- ALL}$

In [ ]:
#@title # $\color{#19ABEA}{\text{Download and install requirements}}$ {display-mode: "form"}

%cd /content
!echo Downloading KoboldCpp, please wait...
!wget -O dlfile.tmp https://kcpplinux.concedo.workers.dev && mv dlfile.tmp koboldcpp_linux
!test -f koboldcpp_linux && echo Download Successful || echo Download Failed
!chmod +x ./koboldcpp_linux
!apt update
!apt install aria2 -y


In [ ]:

#@title #  $\color{#19ABEA}{Download- Model}$ {display-mode: "form"}
#@markdown ## $\color{#19ABEA}{\text{Copy the RepoName here :}}$

# ------------------------------------------
from huggingface_hub import HfFileSystem
fs = HfFileSystem()

def get_donwloadlink_and_filename(repo_name,quant='Q4_K_M'):
  gguf_files = fs.glob(f"{repo_name}/*.gguf")
  filtered_files = [file for file in gguf_files if (quant in file or quant.lower() in file.lower() or quant.upper() in file.upper())]

  if len(filtered_files) ==0 : # quant not existed
    print( "=" * 10 +"\navailable quants:\n",gguf_files)
    return None,None
  file_path= filtered_files[0]
  *repo_name , file_name = file_path.split("/");
  repo_name='/'.join(repo_name)
  return f"https://huggingface.co/{repo_name}/resolve/main/{file_name}?download=true" ,file_name

# ------------------------------------------
#@markdown  Model name
#@markdown  Example :`bartowski/Meta-Llama-3.1-8B-Instruct-GGUFF`
repo_name = "DevQuasar/nvidia.NVIDIA-Nemotron-Nano-12B-v2-GGUF" # @param ["QuantFactory/Meta-Llama-3-8B-Instruct-GGUF","bartowski/gemma-2-9b-it-GGUF","QuantFactory/Mistral-Nemo-Instruct-2407-GGUF","bartowski/Mistral-Small-Instruct-2409-GGUF","bartowski/Meta-Llama-3.1-8B-Instruct-GGUF","unsloth/Qwen3-30B-A3B-GGUF","unsloth/gpt-oss-20b-GGUF","unsloth/Qwen3-8B-GGUF","unsloth/Qwen3-14B-GGUF","unsloth/gemma-3-12b-it-GGUF"] {"allow-input":true}

# ------------------------------------------
# ------------------------------------------
# ------------------------------------------
quant = "Q4_K_M" # @param ["Q2_K","Q3_K_L","Q3_K_M","Q3_K_S","Q4_0","Q4_1","Q4_K_M","Q4_K_S","Q5_0","Q5_1","Q5_K_M","Q5_K_S","Q6_K","Q8_0"]
model_download_url,model_file_name =get_donwloadlink_and_filename(repo_name,quant=quant)

if model_download_url is not None and model_file_name is not None :
  model_download_url=model_download_url.replace("?download=true","")
  print(model_download_url)
  !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {model_download_url} -d /content/ -o {model_file_name}
  # !aria2c -c -x 10 -o {model_file_name} --summary-interval=5 --download-result=default --auto-file-renaming=false --file-allocation=none {model_download_url}
else :
  print("======================")
  print("please check the repo name or check availability of the quantized file inside the repo")
  print("======================")
  print("======================")
  print("======================")
  raise RuntimeError("⚠️Quant doest exist in the repository")



In [ ]:
#@title #  $\color{#19ABEA}{Start- koboldCpp}$ {display-mode: "form"}
Model = model_download_url
#@markdown Note : change the offloaded layer only if the model cant fit in the gpu.

#@markdown - for models like `qwen-30B-A3B` it fits only 31 layer in gpu and the rest in system ram,

#@markdown - for   `gpt-oss-20b` model you can fully load it with `Q4_k_m` .

Layers = "99" # @param ["99","35","31"] {"allow-input":true}
ContextSize = "4096" # @param ["4096","8192","12288","16384"] {"allow-input":true}
FlashAttention = True #@param {type:"boolean"}
Multiplayer = False #@param {type:"boolean"}
FACommand = ""
MPCommand = ""
#@markdown <hr>
LoadVisionMMProjector = False #@param {type:"boolean"}
Mmproj = "https://huggingface.co/koboldcpp/mmproj/resolve/main/LLaMA3-8B_mmproj-Q4_1.gguf" #@param ["https://huggingface.co/koboldcpp/mmproj/resolve/main/llama-13b-mmproj-v1.5.Q4_1.gguf","https://huggingface.co/koboldcpp/mmproj/resolve/main/mistral-7b-mmproj-v1.5-Q4_1.gguf","https://huggingface.co/koboldcpp/mmproj/resolve/main/llama-7b-mmproj-v1.5-Q4_0.gguf","https://huggingface.co/koboldcpp/mmproj/resolve/main/LLaMA3-8B_mmproj-Q4_1.gguf"]{allow-input: true}
VCommand = ""
#@markdown <hr>
LoadImgModel = False #@param {type:"boolean"}
ImgModel = "https://huggingface.co/koboldcpp/imgmodel/resolve/main/imgmodel_ftuned_q4_0.gguf" #@param ["https://huggingface.co/koboldcpp/imgmodel/resolve/main/imgmodel_ftuned_q4_0.gguf"]{allow-input: true}
SCommand = ""
#@markdown <hr>
LoadSpeechModel = False #@param {type:"boolean"}
SpeechModel = "https://huggingface.co/koboldcpp/whisper/resolve/main/whisper-base.en-q5_1.bin" #@param ["https://huggingface.co/koboldcpp/whisper/resolve/main/whisper-base.en-q5_1.bin"]{allow-input: true}
WCommand = ""
#@markdown <hr>
LoadTTSModel = False #@param {type:"boolean"}
TTSModel = "https://huggingface.co/koboldcpp/tts/resolve/main/OuteTTS-0.2-500M-Q4_0.gguf" #@param ["https://huggingface.co/koboldcpp/tts/resolve/main/OuteTTS-0.2-500M-Q4_0.gguf"]{allow-input: true}
WavTokModel = "https://huggingface.co/koboldcpp/tts/resolve/main/WavTokenizer-Large-75-Q4_0.gguf" #@param ["https://huggingface.co/koboldcpp/tts/resolve/main/WavTokenizer-Large-75-Q4_0.gguf"]{allow-input: true}
TTSCommand = ""
#@markdown <hr>
LoadEmbeddingsModel = False #@param {type:"boolean"}
EmbeddingsModel = "https://huggingface.co/yixuan-chia/snowflake-arctic-embed-s-GGUF/resolve/main/snowflake-arctic-embed-s-Q4_0.gguf" #@param ["https://huggingface.co/yixuan-chia/snowflake-arctic-embed-s-GGUF/resolve/main/snowflake-arctic-embed-s-Q4_0.gguf"]{allow-input: true}
ECommand = ""
#@markdown <hr>
#@markdown This enables saving stories directly to your google drive. You will have to grant permissions, and then you can access the saves from the "KoboldCpp Server Storage" option.
AllowSaveToGoogleDrive = False #@param {type:"boolean"}
SavGdriveCommand = ""
#@markdown <hr>
#@markdown Only select the following box if regular cloudflare tunnel fails to work. It will generate an inferior localtunnel tunnel, which you can use after entering a password.
MakeLocalTunnelFallback = False #@param {type:"boolean"}

import os
if not os.path.isfile("/opt/bin/nvidia-smi"):
  raise RuntimeError("⚠️Colab did not give you a GPU due to usage limits, this can take a few hours before they let you back in. Check out https://lite.koboldai.net for a free alternative (that does not provide an API link but can load KoboldAI saves and chat cards) or subscribe to Colab Pro for immediate access.⚠️")

if AllowSaveToGoogleDrive:
  print("Attempting to request access to save to your google drive...")
  try:
    from google.colab import drive
    import os, json
    drive.mount('/content/drive', force_remount=True)
    if not os.path.exists("/content/drive/MyDrive"):
      raise RuntimeError("Google Drive mount failed. Please grant permissions and try again.")
    kcppdir = '/content/drive/MyDrive/koboldcpp_data'
    os.makedirs(kcppdir, exist_ok=True)
    savedatapath = os.path.join(kcppdir, "koboldcpp_save_db.jsondb")
    if not os.path.exists(savedatapath):
        settings_data = {}
        with open(savedatapath, "w") as json_file:
            json.dump(settings_data, json_file, indent=4)
        print(f"Created new koboldcpp_save_db.jsondb at {savedatapath}")
    else:
        print(f"Loading saved data at {savedatapath}")
    SavGdriveCommand = f" --savedatafile {savedatapath}"
  except Exception as e:
    print(f"⚠️ Error: {e}")
    print("Please ensure you grant Google Drive permissions and try again.")

%cd /content
if Mmproj and LoadVisionMMProjector:
  VCommand = "--mmproj vmodel.gguf"
else:
  SCommand = ""
if ImgModel and LoadImgModel:
  SCommand = "--sdmodel imodel.gguf --sdthreads 4 --sdquant --sdclamped"
else:
  SCommand = ""
if SpeechModel and LoadSpeechModel:
  WCommand = "--whispermodel wmodel.bin"
else:
  WCommand = ""
if TTSModel and WavTokModel and LoadTTSModel:
  TTSCommand = "--ttsmodel ttsmodel.bin --ttswavtokenizer ttswavtok.bin --ttsgpu"
else:
  TTSCommand = ""
if EmbeddingsModel and LoadEmbeddingsModel:
  ECommand = "--embeddingsmodel emodel.bin"
else:
  ECommand = ""
if FlashAttention:
  FACommand = "--flashattention"
else:
  FACommand = ""
if Multiplayer:
  MPCommand = "--multiplayer"
else:
  MPCommand = ""
# ---------------------------------------------------------------------------------------------
# !echo Downloading KoboldCpp, please wait...
# !wget -O dlfile.tmp https://kcpplinux.concedo.workers.dev && mv dlfile.tmp koboldcpp_linux
# !test -f koboldcpp_linux && echo Download Successful || echo Download Failed
# !chmod +x ./koboldcpp_linux
# !apt update
# !apt install aria2 -y
# simple fix for a common URL mistake
# ---------------------------------------------------------------------------------------------
if "https://huggingface.co/" in Model and "/blob/main/" in Model:
  Model = Model.replace("/blob/main/", "/resolve/main/")
# !aria2c -x 10 -o {model_file_name} --summary-interval=5 --download-result=default --allow-overwrite=true --file-allocation=none $Model
# download only if diffrent
# !aria2c -c -x 10 -o {model_file_name} --summary-interval=5 --download-result=default --auto-file-renaming=false --file-allocation=none $Model
if VCommand:
  !aria2c -x 10 -o vmodel.gguf --summary-interval=5 --download-result=default --allow-overwrite=true --file-allocation=none $Mmproj
if SCommand:
  !aria2c -x 10 -o imodel.gguf --summary-interval=5 --download-result=default --allow-overwrite=true --file-allocation=none $ImgModel
if WCommand:
  !aria2c -x 10 -o wmodel.bin --summary-interval=5 --download-result=default --allow-overwrite=true --file-allocation=none $SpeechModel
if TTSCommand:
  !aria2c -x 10 -o ttsmodel.bin --summary-interval=5 --download-result=default --allow-overwrite=true --file-allocation=none $TTSModel
  !aria2c -x 10 -o ttswavtok.bin --summary-interval=5 --download-result=default --allow-overwrite=true --file-allocation=none $WavTokModel
if ECommand:
  !aria2c -x 10 -o emodel.bin --summary-interval=5 --download-result=default --allow-overwrite=true --file-allocation=none $EmbeddingsModel

if MakeLocalTunnelFallback:
  import urllib
  print("Trying to use LocalTunnel as a fallback tunnel (not so good)...")
  ltpw = urllib.request.urlopen('https://loca.lt/mytunnelpassword').read().decode('utf8').strip("\n")
  !nohup npx --yes localtunnel --port 5001 > lt.log 2>&1 &
  !sleep 8
  print("=================")
  print("(LocalTunnel Results)")
  !cat lt.log
  print(f"Please open the above link, and input the password '{ltpw}'\nYour KoboldCpp will start shortly...")
  print("=================")
  !sleep 10
# !./koboldcpp_linux model.gguf --usecublas 0 mmq --chatcompletionsadapter AutoGuess --multiuser --gpulayers $Layers --contextsize $ContextSize --websearch --quiet --remotetunnel $FACommand $MPCommand $VCommand $SCommand $WCommand $TTSCommand $ECommand $SavGdriveCommand
!./koboldcpp_linux {model_file_name} --usecublas 0 mmq --chatcompletionsadapter AutoGuess --multiuser --gpulayers $Layers --contextsize $ContextSize --websearch --quiet --remotetunnel $FACommand $MPCommand $VCommand $SCommand $WCommand $TTSCommand $ECommand $SavGdriveCommand
